# Extract point clouds (ground truth, baseline, uncertainty-aware)

Out structure

| Field | Meaning |
|---|---|
| `x, y, z` | coordinates (recentred metres, or absolute with `ABSOLUTE_COORDS`) |
| `gt` | original label values |
| `pred_baseline`, `pred_uncertainty` | the two models' predicted labels |
| `correct_baseline`, `correct_uncertainty` | 1 where the prediction equals `gt` |
| `comparison` | 0 both correct · 1 both wrong · 2 fixed by uncertainty · 3 broken by uncertainty |
| `u_tilde` | the epistemic uncertainty input |

## Setup

In [ ]:
import os, sys
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

from src import paths
print('project folder:', paths.PROJECT_ROOT)

import torch
from src.export import exportDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
DATASETS        = ['ahn3', 'paris_lille', 'toronto3d', 'vaihingen']
SEED_RULE       = 'best_each'
PREDICTIONS     = 'auto' #'auto' (reuse notebook 03's predictions, run the checkpoint if missing), 'infer' (always run the checkpoints), 'saved' (never run them)
FORMATS         = ['ply'] #ply/laz/las or all three could be added
ABSOLUTE_COORDS = False #Restores he original coordinatres
EVAL_OVERRIDES  = []

## Export
Datasets that fail are reported and skipped, the others still run.

In [ ]:
results, errors = {}, []
for ds in DATASETS:
    print(f'\n===== {ds} =====')
    try:
        results[ds] = exportDataset(ds, SEED_RULE, PREDICTIONS, FORMATS, ABSOLUTE_COORDS, device, EVAL_OVERRIDES)
    except Exception as e:
        print(f'[{ds}] FAILED: {e}')
        errors.append({'dataset': ds, 'error': str(e)})
    if device.type == 'cuda':
        torch.cuda.empty_cache()
if errors:
    display(pd.DataFrame(errors))

## Chosen runs
The runs each file was built from, with their test metrics from notebook 03

In [ ]:
if not results:
    raise SystemExit('nothing was exported; see the errors above')
rows = []
for ds, (info, stats, table) in results.items():
    for v in ('baseline', 'uncertainty'):
        m = info['metrics'][v]
        rows.append({'dataset': ds, 'variant': v, 'seed': info['seeds'][v], 'OA_all': m['OA_all'],
                     'mIoU': m['mIoU'], 'PAvPU': m['PAvPU'], 'predictions': info['predictionSource'][v]})
display(pd.DataFrame(rows).round(4))

## What changed between the two models
Share of test points in each comparison category. "Fixed" and "broken" are the points where the two models disagree on correctness

In [ ]:
allStats = pd.concat([stats.assign(dataset=ds) for ds, (_, stats, _) in results.items()], ignore_index=True)
cols = ['dataset', 'file', 'points', 'OA_baseline', 'OA_uncertainty', 'pct_both_correct', 'pct_both_wrong',
        'pct_fixed_by_unc', 'pct_broken_by_unc']
display(allStats[cols])
for ds, (info, _, _) in results.items():
    print(f'\n{ds}:')
    for p in info['files']:
        print('   ', p)